# Profiling experiment

Profiles one model under several settings, each profile an artifact the launcher runs on its own GPU, its files kept on the volume. Run it in the lab: `create_experiment` needs the volume's mount. Creating the experiment keeps this notebook, as last saved, at `experiments/<experiment_uid>/declaration.ipynb`, and in each profile's folder as `declaring.ipynb`.

In [ ]:
import lab
from artifacts.core.artifact import Resources
from training.optimizers import SGD
from artifacts.profiles import Profile
from models.transformer import Transformer

import pandas as pd


def transformer_by_size(size: str) -> Transformer:
    """Returns the Transformer shaped as row `size` of `sizes`."""
    return Transformer(
        vocab_size=vocab_size,
        sequence_length=context_length,
        rope_theta=10000,
        dtype="torch.float32",
        **models.loc[size].to_dict(),
    )

models = pd.DataFrame(
    [
        ("small", 768, 3072, 12, 12),
        ("medium", 1024, 4096, 24, 16),
        ("large", 1280, 5120, 36, 20),
        ("xl", 2560, 10240, 32, 32),
        ("10B", 4608, 12288, 50, 36),
    ],
    columns=["size", "d_model", "d_ff", "num_layers", "num_heads"],
).set_index("size")

vocab_size = 10000
context_length = 512
batch_size = 4
models

,d_model,d_ff,num_layers,num_heads
size,,,,
small,768,3072,12,12
medium,1024,4096,24,16
large,1280,5120,36,20
xl,2560,10240,32,32
10B,4608,12288,50,36


## Sweep

One profile per `d_model`, everything else equal.

In [2]:
experiment_uid = "profile-compile"

profiles = [
    Profile(
        run_id=f"{experiment_uid}-mode_{mode}",
        resources=Resources(gpu_type="A10"),
        model=transformer_by_size('small'),
        optimizer=SGD(),
        step_kind="full",  # or "forward": the forward alone, without gradients
        batch_size=4,
        seed=0,
        warmup_steps=5,  # enough for a compiled step to finish compiling
        profile_steps=10,
        autocast=None,  # or "torch.bfloat16": what forward and loss run in; None runs in the model's dtype
        compile=mode,  # a torch.compile mode ("default", "max-autotune"), or None to run eagerly
        kernel_events=False,
        module_scopes=False,  # must be False with compile
        peak_memory=True,
        trace_memory=False,
        tags={"experiment": [experiment_uid]},
    )
    for mode in (None,"default","max-autotune")
]
for profile in profiles:
    print(profile.artifact_path)

profiles/profile-compile-mode_None/3f194dbeda
profiles/profile-compile-mode_default/e5fd5f93d4
profiles/profile-compile-mode_max-autotune/b9d44aa451


## Create

Save the notebook first: the copy kept is the file on disk, and a notebook whose saved file lacks this cell is refused. `commit=False` previews, writing nothing. To add profiles to an experiment that exists, pass `extend="add"`, which keeps this notebook as the next `declaration<n>.ipynb` beside the first, or `extend="overwrite"`, which replaces `declaration.ipynb`.

In [3]:
report = lab.create_experiment(experiment_uid, profiles, verbose=True, commit=False)

RuntimeError: reload() can only be called from within a running function

## Read back

Once the launcher has run them, `load_experiment` lists the experiment's profiles and `bind` reads each one's report; events and memory are read when a method needs them. From a laptop, copy each `run_id`'s folder into an existing `storage/profiles` first, `modal volume get --force nlp profiles/<run_id> storage/profiles`, and pass that `storage` folder as `root` to `load_experiment`, `status` and `bind`.

In [ ]:
bound = [profile.bind() for profile in lab.load_experiment(experiment_uid) if profile.status().complete]
table = pd.concat([profile.summary() for profile in bound])
table.pivot_table("mfu", index="model.d_model", columns="step")

In [ ]:
bound[0].display_steps(steps=[1])